# 01 · 배포 버전과 실행 설정

목표: 같은 코드의 설정 차이를 식별하고 시작·준비·종료 상태를 확인한다. 약12분.

**이 파일의 핵심 질문:** 어떤 설정이 바뀌었으며, 현재 동작을 어떤 버전·설정으로 재현할 수 있는가?

**manifest**는 실행 버전·설정 목록, **config_id**는 허용된 설정 값의 지문이다. **lifespan**은 앱 시작·종료 처리, **readiness**는 새 요청을 받을 준비 상태다.

**수업 중 먼저 실행할 단계**

- 1 기본 설정
- 4 앱의 수명주기
- 5 버전 응답

파일 안의 변수 의존성이 있으므로 처음 실행할 때는 위에서부터 순서대로 진행한다. 핵심 단계에서는 출력을 읽고, 나머지 단계는 빠르게 실행한 뒤 복습 때 살펴본다.

## 시작 전 · 어디에서 무엇을 실행하나요?

1. VS Code에서 `llmops-16week` **폴더 전체**를 연다. 압축을 풀었다면 Notebook 파일만 따로 옮기지 않는다.
2. VS Code의 Python·Jupyter 확장을 설치하고 우측 위 **Select Kernel → Python Environments → .venv / Python 3.11**을 선택한다. 터미널의 가상환경과 Notebook 커널은 별도 선택이다.
3. 아래 **환경 확인 코드 셀부터** `Shift+Enter`로 한 셀씩 실행한다. 이 파일 안에서는 앞에서 만든 변수를 뒤에서 사용한다.
4. 각 단계의 **예상 결과**와 실제 출력을 대조한다. `assert`는 확인 조건이다. 출력이나 오류가 없어도 셀이 완료되면 조건을 통과한 것이다.
5. 셀을 건너뛰어 `NameError`가 나거나 변수를 여러 번 바꿨다면 **Restart Kernel → Run All**로 처음부터 실행한다. 다른 Notebook의 변수는 가져오지 않는다.

처음 환경을 만드는 경우에만 **VS Code 터미널**에서 실행한다. 아래 명령을 Python 코드 셀에 붙여넣지 않는다.

```bash
# macOS / Linux · 저장소 루트
python3.11 -m venv .venv
source .venv/bin/activate
python -m pip install -r requirements.txt
```

Windows PowerShell은 `py -3.11 -m venv .venv`, `.venv\Scripts\Activate.ps1` 순서다. 활성화가 제한된 환경에서는 `.venv\Scripts\python.exe -m pip install -r requirements.txt`처럼 가상환경 Python을 직접 실행한다. 이미 준비된 `.venv`는 그대로 쓴다.

**공통 실습의 실행 방식:** `lab_client`가 메모리 안에 FastAPI 앱을 만들고 시작·종료를 관리한다. HTTPX의 ASGITransport가 같은 프로세스에서 요청을 전달하므로 `uvicorn`, Docker, Ollama, API 키가 필요하지 않다. ASGI는 *Asynchronous Server Gateway Interface*, Python 비동기 웹 앱과 서버 사이의 인터페이스다.

Notebook 앱과 브라우저의 `127.0.0.1:8010` 앱은 **서로 다른 실행**이다. Notebook에서 만든 요청 수가 브라우저·Prometheus에 합쳐지지 않는다. 실제 모델 대신 고정 문장과 대기를 쓰므로 외부 비용은 발생하지 않는다. 실행 시간·요청 ID는 매번 달라져도 정상이다.

In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "week10/lab/app.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import asyncio, json
from dataclasses import replace
from week10.lab.config import Settings
from week10.lab.experiments import lab_client, fixture_check, window_delta, compare_releases
assert sys.version_info[:2] == (3, 11), "Select the Python 3.11 .venv kernel in VS Code"
print("Python:", sys.version.split()[0])
print("실행 환경:", sys.executable)
print("저장소:", ROOT)
print("환경 확인 완료 · 이 Notebook은 별도 서버 없이 실행합니다.")

## 1. 기본 설정

릴리스·프롬프트·모델·데이터가 따로 기록되는 이유를 설명해본다.

**예상 결과:** release=v1, model=mock-fixed-v1, timeout_s=0.25, capacity=2 등이 담긴 사전.

**결과 해석:** 같은 코드라도 프롬프트·데이터·처리 한도가 다르면 실행 조건이 다르다. 비교 전에 이 목록을 남긴다.

In [ ]:
cfg = Settings()
manifest = cfg.manifest()
manifest

## 2. 설정의 동일성

같은 입력은 같은 config_id다. 코드 commit이나 이미지 digest를 대신하는 값은 아니다.

**예상 결과:** 기본 설정의 manifest는 같고, timeout_s를0.5로 바꾸면 config_id가 달라진다.

**결과 해석:** 설정 차이를 찾는 식별자다. 코드 변경을 식별하는 Git commit과는 별도로 기록한다.

In [ ]:
assert manifest == Settings().manifest()
changed = replace(cfg, timeout_s=0.5)
assert changed.manifest()["config_id"] != manifest["config_id"]
changed.manifest()

## 3. 시작 전 실패

오타·NaN·과도한 처리량을 조용히 수용하지 않는다.

**예상 결과:** ValueError 메시지3개. 이 단계의 오류 메시지는 예상한 출력이다.

**결과 해석:** 지원하지 않는 버전·숫자가 아닌 제한시간·0인 처리량을 시작 전에 거부한다. 허용 범위를 바꿔 운영 권장값으로 사용하지 않는다.

In [ ]:
for values in [{"release":"latest"}, {"timeout_s":float("nan")}, {"capacity":0}]:
    try:
        Settings(**values)
    except ValueError as e:
        print(type(e).__name__, str(e))
    else:
        raise AssertionError("invalid configuration accepted")

## 4. 앱의 수명주기

ASGITransport는 startup을 자동 실행하지 않는다. lab_client가 lifespan을 명시적으로 연다.

**예상 결과:** 컨텍스트 안에서는 live·ready가200, 종료 뒤 started=False·draining=True. assert만 있는 부분은 무출력 통과.

**결과 해석:** app 객체 생성만으로 시작 처리가 실행되지는 않는다. lab_client의 async with가 수명주기를 열고 닫는다.

In [ ]:
from week10.lab.app import create_app
app = create_app(cfg)
assert not app.state.started
async with lab_client(cfg) as (running, client):
    print((await client.get("/health/live")).json())
    assert (await client.get("/health/ready")).status_code == 200
assert not running.state.started and running.state.draining

## 5. 버전 응답

요청 결과의 release와 서버 manifest를 대조한다.

**예상 결과:** HTTP200 응답, release=v1, simulated=true. X-Release와 /release의 값이 일치한다.

**결과 해석:** 학생은 화면의 이름만 보지 않고 실제 처리 버전을 응답에서도 확인한다. ID·실행 시간은 매번 달라진다.

In [ ]:
async with lab_client(cfg) as (app, client):
    r = await client.post("/api/v1/assist", json={"message":"신청 절차"})
    assert r.headers["X-Release"] == (await client.get("/release")).json()["release"]
    print(r.json())

## 6. 컨테이너 실행 경로

파일을 읽기만 한다. Docker 설치·다운로드·데몬 실행을 자동 수행하지 않는다.

**예상 결과:** Dockerfile 본문과 USER 10001 설정.

**결과 해석:** 파일 읽기만 실행한다. 이미지 빌드·컨테이너 실행은 터미널의 선택 실습이다.

In [ ]:
dockerfile = (ROOT / "week10/lab/Dockerfile").read_text()
print(dockerfile)
assert "USER 10001" in dockerfile and "--host" in dockerfile

## 7. 내 설정의 경계

동시 처리량을1·2로 바꿔 manifest 차이를 확인한다. 이 단계는 설정 비교다.

**예상 결과:** capacity=1·2인 manifest 두 개와 서로 다른 config_id.

**결과 해석:** capacity는 동시에 받아들일 작업 수의 상한이다. 이 셀은 설정 비교이며 처리량 성능 측정은 아니다.

In [ ]:
for cap in [1, 2]:
    print(Settings(capacity=cap).manifest())

## 직접 바꿔볼 실험 · 선택 복습

1. timeout_s를0.25와0.5로 비교하고 config_id가 바뀐 이유를 적는다.
2. 정상 범위의 capacity=1과2를 비교한 뒤 원래 코드로 복원한다.
3. 실제 프로젝트에서 manifest에 넣을 값3개와 공개하면 안 되는 값1개를 고른다.

한 번에 한 조건만 바꾸고 원래 코드와 비교한다. 기대값이 고정된 assert가 있다면 새 실험 조건에 맞는 값도 함께 설명한다.

## 프로젝트 적용 기록

**정리 질문:** 어떤 설정이 바뀌었으며, 현재 동작을 어떤 버전·설정으로 재현할 수 있는가?

아래 Markdown 셀을 더블클릭해 자신의 결과를 적는다. 다른 파일 제출이나 화면 실행을 자동으로 수행하지 않는다.

| 항목 | 직접 기록 |
|---|---|
| 예상 | 실행 전 예상한 결과 |
| 관찰 | 상태 코드·건수·실패 단계 등 실제 결과 |
| 근거 | 실행한 단계 번호·변경한 조건·캡처 위치 |
| 판단 | 유지·수정·채택·보류 이유 |
| 미검증·다음 변경 | 아직 실행하지 않은 조건과 다음 비교 |

학생 정보나 원문 입력은 공개 저장소에 올리지 않는다. 실제 실행하지 않은 조건은 미측정으로 표시한다.

## 막혔을 때의 확인 순서

| 증상 | 확인·해결 |
|---|---|
| 커널 선택 창만 표시 | Python·Jupyter 확장 설치 후 `.venv`의 Python 선택 |
| `ModuleNotFoundError` | 커널 경로 확인 → 해당 `.venv`로 `python -m pip install -r requirements.txt` |
| 환경 확인에서 저장소를 찾지 못함 | 저장소 전체를 열고 원래 `week10/lab/notebooks` 위치에서 실행 |
| `NameError` | 앞 셀 실행 여부 확인 → Restart Kernel → Run All |
| `asyncio.run() cannot be called...` | Jupyter에서는 제공된 최상위 `await`·`async with`를 그대로 사용 |
| 셀에 출력이 없음 | `assert`만 있는 셀은 오류 없이 끝나면 통과. 왼쪽 실행 번호와 완료 표시 확인 |
| 예상한 422·502·503·504 | 입력 오류·검색 오류·준비 실패·시간 초과를 의도적으로 만든 단계인지 확인 |
| 예상과 다른 `AssertionError` | 바로 위 코드·출력 확인. 변경한 값을 복원하고 처음부터 다시 실행. assert를 지워 통과시키지 않음 |
| 실행 시간·ID가 예시와 다름 | 고정값 비교 대신 상태·부모/자식 관계·분모·검증 결과를 비교 |

실습에서 바꾼 내용은 자신의 사본에 기록한다. 공개 저장소에는 학생 정보·원문 질문·실제 키를 올리지 않는다.